In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# AI搭載Transpilerパス

AI搭載Transpilerパスは、一部のトランスパイル処理において「従来の」Qiskitパスのドロップイン置き換えとして機能するパスです。既存のヒューリスティックアルゴリズムよりも優れた結果（低い深度やCNOTカウントなど）を生成することが多く、ブール充足可能性ソルバーなどの最適化アルゴリズムよりもはるかに高速です。AI Transpilerパスはローカル環境で実行されます。

> **Note:** AI搭載Transpilerパスはベータリリース状態であり、変更される可能性があります。
> フィードバックや開発チームへの連絡は、こちらの[Qiskit Slack Workspaceチャンネル](https://qiskit.slack.com/archives/C06KF8YHUAU)を使ってください。

現在、以下のパスが利用可能です：

**ルーティングパス**
 - `AIRouting`: レイアウト選択と回路ルーティング

**回路合成パス**
 - `AICliffordSynthesis`: Clifford回路合成
 - `AILinearFunctionSynthesis`: 線形関数回路合成
 - `AIPermutationSynthesis`: 置換回路合成

AI Transpilerパスを使用するには、まず`qiskit-ibm-transpiler`パッケージをインストールしてください。利用可能なさまざまなオプションの詳細については、[qiskit-ibm-transpiler APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)を参照してください。

## AI Transpilerパスをローカルまたはクラウドで実行する
まず、追加の依存関係を含めて`qiskit-ibm-transpiler`を以下のようにインストールしてください：

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

追加依存関係をインストールした後は、AI搭載Transpilerパスのデフォルトの実行モードはローカルマシンの使用になります。

## AIルーティングパス
`AIRouting`パスは、レイアウトステージとルーティングステージの両方として機能します。以下のように`PassManager`内で使用できます：

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

ここで、`backend`はルーティング対象のカップリングマップを決定し、`optimization_level`（1、2、または3）は処理に費やす計算量を決定し（高いほど通常は良い結果が得られますが、時間がかかります）、`layout_mode`はレイアウト選択の処理方法を指定します。
`layout_mode`には以下のオプションがあります：

- `keep`: 前のTranspilerパスで設定されたレイアウトを尊重します（設定されていない場合はトリビアルレイアウトを使用します）。通常、デバイスの特定のQubitで回路を実行する必要がある場合にのみ使用されます。最適化の余地が少ないため、結果が悪くなることが多いです。
- `improve`: 前のTranspilerパスで設定されたレイアウトを開始点として使用します。レイアウトの良い初期推定がある場合に便利です。例えば、デバイスのカップリングマップにおおよそ従うように構築された回路の場合です。また、他の特定のレイアウトパスを`AIRouting`パスと組み合わせて試したい場合にも便利です。
- `optimize`: デフォルトモードです。レイアウトの良い推定がない一般的な回路に最適です。このモードは前のレイアウト選択を無視します。
- `local_mode`: このフラグは`AIRouting`パスの実行場所を決定します。`False`の場合、`AIRouting`はQiskit Transpiler Serviceを通じてリモートで実行されます。`True`の場合、パッケージはローカル環境でパスを実行しようとし、必要な依存関係が見つからない場合はクラウドモードにフォールバックします。

## AI回路合成パス
AI回路合成パスを使用すると、さまざまな回路タイプ（[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford)、[Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction)、[Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation)、Pauli Network）の部分を再合成して最適化できます。合成パスの典型的な使用方法は以下の通りです：